# Two-Mode Frequency Gap Sweep: From 2-Torus to Circle

Sweeps the frequency gap Δω between two AR(4) oscillatory modes and measures how torus geometry degrades as modes merge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from scipy.signal import welch

np.random.seed(42)
%matplotlib inline

## Helpers

In [ ]:
def make_ar4_coeffs(omega1, omega2, r=0.98):
    """AR(4) from two conjugate pole pairs at frequencies omega1, omega2."""
    a1 = np.array([1, -2 * r * np.cos(omega1), r**2])
    a2 = np.array([1, -2 * r * np.cos(omega2), r**2])
    poly = np.convolve(a1, a2)
    return -poly[1:]


def simulate_ar(phi, N=6000, sigma=0.03):
    p = len(phi)
    x = np.zeros(N)
    x[:p] = np.random.randn(p) * 0.1
    for t in range(p, N):
        x[t] = np.dot(phi, x[t - p:t][::-1]) + sigma * np.random.randn()
    return x


def lag_embed_3d(x, tau):
    """3D lag embedding: [x(t), x(t-tau), x(t-2tau)]."""
    N = len(x)
    M = N - 2 * tau
    return np.column_stack([x[2 * tau:], x[tau:tau + M], x[:M]])


def fit_torus(emb):
    """
    Fit a torus to 3D point cloud.
    Torus centered at origin in xy-plane.
    """
    pts = emb - emb.mean(axis=0)
    scale = np.std(pts) + 1e-8
    pts /= scale

    xy = np.sqrt(pts[:, 0]**2 + pts[:, 1]**2)
    z = pts[:, 2]

    R_vals = np.linspace(0.1, 2.5, 400)
    best_R, best_mse = 1.0, np.inf

    for R in R_vals:
        d = np.sqrt((xy - R)**2 + z**2)
        r = d.mean()
        mse = np.mean((d - r)**2)
        if mse < best_mse:
            best_mse = mse
            best_R = R

    d_best = np.sqrt((xy - best_R)**2 + z**2)
    r_best = d_best.mean()
    torus_score = float(np.mean(np.abs(d_best - r_best) < r_best))
    return torus_score, best_mse

## Parameters

In [ ]:
omega_base = 0.20
tau = 6

N = 6000
sigma = 0.03
n_gaps = 30
n_rep = 6

delta_omegas = np.linspace(0.02, 0.40, n_gaps)

## Full sweep for summary curve

In [ ]:
all_scores = np.zeros((n_rep, n_gaps))
all_mse = np.zeros((n_rep, n_gaps))

print("Running sweep for summary curve ...")
for j, dw in enumerate(delta_omegas):
    for rep in range(n_rep):
        np.random.seed(rep * 200 + j)
        phi = make_ar4_coeffs(omega_base, omega_base + dw)
        x = simulate_ar(phi, N=N, sigma=sigma)
        emb = lag_embed_3d(x, tau)
        ts, mse = fit_torus(emb)
        all_scores[rep, j] = ts
        all_mse[rep, j] = mse

mean_s = all_scores.mean(0)
std_s = all_scores.std(0)
mean_m = all_mse.mean(0)
std_m = all_mse.std(0)

print("Done.")

## Representative cases

In [ ]:
cases = [
    dict(dw=0.38, label="Well-separated modes",   color="#4C78A8"),
    dict(dw=0.15, label="Partially merged modes", color="#B2792A"),
    dict(dw=0.03, label="Nearly merged modes",    color="#A05144"),
]

for c in cases:
    np.random.seed(7)
    phi = make_ar4_coeffs(omega_base, omega_base + c["dw"])
    x = simulate_ar(phi, N=N, sigma=sigma)
    emb = lag_embed_3d(x, tau)
    ts, ms = fit_torus(emb)
    c["x"] = x
    c["emb"] = emb
    c["ts"] = ts
    c["ms"] = ms
    print(f"Δω={c['dw']:.2f}  torus_score={ts:.3f}  mse={ms:.4f}")

## Figure

In [ ]:
fig = plt.figure(figsize=(14, 13))
fig.patch.set_facecolor("white")

outer = gridspec.GridSpec(
    4, 1, figure=fig,
    hspace=0.55,
    top=0.94, bottom=0.06,
    left=0.07, right=0.97
)

top_gs = gridspec.GridSpecFromSubplotSpec(
    3, 3, subplot_spec=outer[0:3],
    hspace=0.55, wspace=0.38
)

bottom_gs = gridspec.GridSpecFromSubplotSpec(
    1, 2, subplot_spec=outer[3],
    wspace=0.35
)

fs = 2000

for row, c in enumerate(cases):
    dw = c["dw"]
    color = c["color"]
    x = c["x"]
    emb = c["emb"]

    # ── col 0 : power spectrum ────────────────────────────────────────────
    ax_psd = fig.add_subplot(top_gs[row, 0])
    f_hz, Pxx = welch(x, fs=fs, nperseg=1024)

    f1_hz = omega_base / (2 * np.pi) * fs
    f2_hz = (omega_base + dw) / (2 * np.pi) * fs

    ax_psd.semilogy(f_hz, Pxx, color=color, lw=1.8)
    ax_psd.axvline(f1_hz, color="k", lw=1.0, ls="--", alpha=0.55)
    ax_psd.axvline(f2_hz, color="k", lw=1.0, ls=":", alpha=0.55)

    ax_psd.set_xlim(0, fs * 0.25)
    ax_psd.set_xlabel("Frequency (Hz)", fontsize=8)
    ax_psd.set_ylabel("PSD (log)", fontsize=8)
    ax_psd.set_title(
        c["label"] + f"\nΔω = {dw:.2f} rad/sample",
        fontsize=9, fontweight="bold", color=color
    )
    ax_psd.tick_params(labelsize=7)
    ax_psd.spines["top"].set_visible(False)
    ax_psd.spines["right"].set_visible(False)

    # ── col 1 : 3D lag embedding ──────────────────────────────────────────
    ax3d = fig.add_subplot(top_gs[row, 1], projection="3d")
    n_plot = 800
    e = emb[:n_plot]

    ax3d.plot(
        e[:, 0], e[:, 1], e[:, 2],
        "-", lw=0.8, alpha=0.80, color=color
    )

    ax3d.set_xlabel("x(t)", fontsize=7, labelpad=1)
    ax3d.set_ylabel("x(t-τ)", fontsize=7, labelpad=1)
    ax3d.set_zlabel("x(t-2τ)", fontsize=7, labelpad=1)
    ax3d.tick_params(labelsize=5)
    ax3d.set_title("Lag embedding", fontsize=9, fontweight="bold")
    ax3d.grid(False)

    ax3d.xaxis.pane.fill = False
    ax3d.yaxis.pane.fill = False
    ax3d.zaxis.pane.fill = False

    # ── col 2 : geometry metrics ──────────────────────────────────────────
    ax_bar = fig.add_subplot(top_gs[row, 2])

    ax_bar.barh(
        ["Torus\nScore"], [c["ts"]],
        color=color, alpha=0.80, height=0.4
    )
    ax_bar.barh(
        ["Torus\nFit MSE"], [c["ms"]],
        color=color, alpha=0.35, height=0.4
    )

    ax_bar.set_xlim(0, 1.0)
    ax_bar.axvline(0.5, color="gray", lw=1.0, ls="--", alpha=0.5)
    ax_bar.set_xlabel("Score / MSE", fontsize=8)
    ax_bar.set_title("Geometry metrics", fontsize=9, fontweight="bold")
    ax_bar.tick_params(labelsize=8)

    ax_bar.text(c["ts"] + 0.02, 1.0, f"{c['ts']:.2f}",
                va="center", fontsize=8, fontweight="bold")
    ax_bar.text(c["ms"] + 0.02, 0.0, f"{c['ms']:.3f}",
                va="center", fontsize=8)

    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)

# ── bottom left : Torus Score vs Δω ──────────────────────────────────────────
ax_sum1 = fig.add_subplot(bottom_gs[0, 0])

ax_sum1.fill_between(
    delta_omegas, mean_s - std_s, mean_s + std_s,
    alpha=0.18, color="#4C78A8"
)
ax_sum1.plot(delta_omegas, mean_s, "o-", color="#4C78A8", lw=2.0, ms=4)

for c in cases:
    ax_sum1.axvline(
        c["dw"], color=c["color"], lw=1.5, ls="--", alpha=0.8,
        label=f"Δω={c['dw']:.2f}"
    )

ax_sum1.set_xlabel("Frequency gap Δω (rad / sample)", fontsize=10)
ax_sum1.set_ylabel("Torus Score (0–1)", fontsize=10)
ax_sum1.set_title("Torus Score vs. Frequency Gap", fontsize=11, fontweight="bold")
ax_sum1.set_xlim(delta_omegas[0], delta_omegas[-1])
ax_sum1.set_ylim(0.95, 1.0)
ax_sum1.legend(fontsize=8, framealpha=0.75)
ax_sum1.grid(True, alpha=0.3)
ax_sum1.spines["top"].set_visible(False)
ax_sum1.spines["right"].set_visible(False)

# ── bottom right : Torus Fit MSE vs Δω ───────────────────────────────────────
ax_sum2 = fig.add_subplot(bottom_gs[0, 1])

ax_sum2.fill_between(
    delta_omegas, mean_m - std_m, mean_m + std_m,
    alpha=0.18, color="#A05144"
)
ax_sum2.plot(delta_omegas, mean_m, "s-", color="#A05144", lw=2.0, ms=4)

for c in cases:
    ax_sum2.axvline(c["dw"], color=c["color"], lw=1.5, ls="--", alpha=0.8)

ax_sum2.set_xlabel("Frequency gap Δω (rad / sample)", fontsize=10)
ax_sum2.set_ylabel("Torus Fit MSE", fontsize=10)
ax_sum2.set_title("Torus Fit MSE vs. Frequency Gap", fontsize=11, fontweight="bold")
ax_sum2.set_xlim(delta_omegas[0], delta_omegas[-1])
ax_sum2.grid(True, alpha=0.3)
ax_sum2.spines["top"].set_visible(False)
ax_sum2.spines["right"].set_visible(False)

fig.suptitle(
    "Two-Mode Frequency Gap Sweep: From 2-Torus to Circle",
    fontsize=13, fontweight="bold", y=0.97
)

plt.savefig("freq_gap_sweep.png", dpi=300, bbox_inches="tight")
plt.savefig("freq_gap_sweep.pdf", dpi=300, bbox_inches="tight")
plt.show()